<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/06_laboratorio_de_muestreo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 06 · Laboratorio de muestreo: temperatura, top-k y top-p

**Objetivo:** entender cómo un modelo **elige** la siguiente palabra.

En cada paso, el modelo calcula una probabilidad para **cada** token del vocabulario. Después hay que
elegir uno:

- **Greedy** (temperatura 0): siempre el más probable → repetitivo.
- **Temperatura**: < 1 hace la distribución más «puntiaguda» (conservador); > 1 la aplana (creativo, caótico).
- **Top-k**: solo se sortea entre los *k* tokens más probables.
- **Top-p** (*nucleus sampling*): solo entre los tokens más probables que suman una probabilidad *p*.

Usamos el GPT diminuto del cuaderno 03 (si no existe, se carga SmolLM2-135M).

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro)
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio e instalamos el paquete del taller
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -e .
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config
cfg = load_config()
INICIO = "Dijo don Quijote"
USAR_MODELO = "auto"   # "auto", "tiny_gpt" o un nombre de Hugging Face, p. ej. "HuggingFaceTB/SmolLM2-135M"

In [ ]:
from pathlib import Path
from dhllm.pretrain import load_tiny_gpt
from dhllm.finetune import load_base_model

ruta = Path(cfg.tiny_gpt.save_dir)
if USAR_MODELO in ("auto", "tiny_gpt") and (ruta / "model.pt").exists():
    modelo, tok = load_tiny_gpt(ruta)
    print("Usando el GPT diminuto del cuaderno 03")
else:
    nombre = "HuggingFaceTB/SmolLM2-135M" if USAR_MODELO in ("auto", "tiny_gpt") else USAR_MODELO
    modelo, tok = load_base_model(nombre)

## 1. ¿Qué token viene después?
Estas son las probabilidades del modelo para el siguiente token.

In [ ]:
from dhllm.sampling import next_token_table
next_token_table(modelo, tok, INICIO, temperature=1.0, n=15);

## 2. El efecto de la temperatura
Misma frase, distinta temperatura. Fíjate en cómo cambian las barras.

In [ ]:
for t in [0.3, 1.0, 2.0]:
    next_token_table(modelo, tok, INICIO, temperature=t, n=10)

## 3. Generar con distintos ajustes

In [ ]:
from dhllm.sampling import compare_settings
compare_settings(modelo, tok, INICIO, [
    {"temperature": 0},                                # greedy
    {"temperature": 0.7, "top_k": 50, "top_p": 0.95},  # equilibrado
    {"temperature": 1.5, "top_k": None, "top_p": 1.0}, # caótico
    {"temperature": 1.0, "top_k": 5, "top_p": 1.0},    # solo 5 opciones
], max_new_tokens=cfg.sampling.max_new_tokens // 2);

## 4. Tu propio experimento

In [ ]:
from dhllm.sampling import generate
print(generate(modelo, tok, INICIO,
               max_new_tokens=cfg.sampling.max_new_tokens,
               temperature=0.9,   # ← cambia
               top_k=40,          # ← cambia (None = sin límite)
               top_p=0.9,         # ← cambia (1.0 = sin límite)
               seed=1))

## 🖐️ Tu turno
1. Encuentra los ajustes que producen el texto más «cervantino». ¿Y el más absurdo?
2. Con `temperature=0`, ¿se repite el texto? ¿Por qué?
3. Ejecuta dos veces con `seed=None`. ¿Por qué cambia el resultado?
4. **Para pensar:** un chatbot responde distinto a la misma pregunta. ¿Qué implica esto para citar
   o usar textos generados en investigación?